### Internações hospitalares — SIH/SUS

SIH/SUS: RD (AIH reduzida), competencias de PROCESSAMENTO 2020–2024.
Notebook Windows 64 bits, Python 3.9–3.13:
  %pip install pyreaddbc==2.0.4 dbfread pyarrow
  %run baixar_sih_sus_2020_2024.py
Fonte: ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/
Escopo: todas as UFs, todos os meses, todos os diagnosticos, somente grupo RD.
Nao e recorte por data de internacao: registros processados em 2025 ficam fora.
AIHs/registros nao equivalem automaticamente a pacientes ou internacoes unicas.
Colunas originais preservadas como texto: codigos com zeros, numeros e datas.
Adiciona origem, competencia, UF do arquivo e flag de exclusao fisica do DBF.
Nao deduplica, nao filtra COVID, nao aplica regras epidemiologicas.
DBC preservado; DBF temporario removido apenas apos conversao validada.
Reexecutar retoma os arquivos concluidos, validando hash e numero de linhas.
O catalogo e uma fotografia da primeira execucao. Para atualizar, use nova PASTA.
Nao execute duas instancias na mesma pasta.

In [2]:
import csv
import ftplib
import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
import pyreaddbc
from dbfread import DBF
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_sih_sus'
ANOS = tuple(range(2020,2025))
MESES = tuple(range(1,13))
UFS = 'AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO'.split()
HOST = 'ftp.datasus.gov.br'
DIRETORIO = '/dissemin/publicos/SIHSUS/200801_/Dados'
LOTE = 20000
TIMEOUT = 60

In [4]:
def salvar_json(p, obj):
    tmp = p.with_suffix(p.suffix+'.part')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8'); tmp.replace(p)


def sha256(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024**2),b''): h.update(b)
    return h.hexdigest()


def conectar():
    ftp=ftplib.FTP(timeout=TIMEOUT)
    try:
        ftp.connect(HOST); ftp.login(); ftp.set_pasv(True); ftp.cwd(DIRETORIO); ftp.voidcmd('TYPE I')
        return ftp
    except Exception:
        ftp.close(); raise


def catalogo():
    cache=PASTA/'catalogo.json'
    if cache.exists():
        obj=json.loads(cache.read_text(encoding='utf-8'))
        if obj.get('host')!=HOST or obj.get('diretorio')!=DIRETORIO: raise ValueError('Catalogo de outra fonte.')
        return obj['arquivos']
    for tentativa in range(1,4):
        ftp=None
        try:
            print(f'Consultando catalogo FTP ({tentativa}/3)...',flush=True)
            ftp=conectar()
            nomes=sorted(set(n.rsplit('/',1)[-1] for n in ftp.nlst()))
            nomes=[n for n in nomes if re.fullmatch(r'RD[A-Z]{2}\d{4}(?:_\d+)?\.dbc',n,re.I)]
            if not nomes: raise ValueError('Nenhum arquivo RD no catalogo.')
            salvar_json(cache,{'host':HOST,'diretorio':DIRETORIO,'consulta_utc':datetime.now(timezone.utc).isoformat(),'arquivos':nomes})
            return nomes
        except Exception as e:
            print(f'Catalogo: {type(e).__name__}: {e}',flush=True)
            if tentativa==3: raise RuntimeError('Nao foi possivel consultar o FTP oficial. Nenhuma coleta completa foi declarada. Verifique acesso FTP (porta 21 e modo passivo) na sua rede e tente novamente.') from e
            time.sleep(3*tentativa)
        finally:
            if ftp: ftp.close()


def selecionar(nomes):
    itens=[]; grupos={}
    for nome in nomes:
        m=re.fullmatch(r'RD([A-Z]{2})(\d{2})(\d{2})(?:_(\d+))?\.dbc',nome,re.I)
        if not m: continue
        uf,yy,mm,parte=m.groups(); ano=2000+int(yy); mes=int(mm); uf=uf.upper()
        if uf not in UFS or ano not in ANOS or mes not in MESES: continue
        grupos.setdefault((uf,ano,mes),[]).append(parte)
        itens.append({'nome':nome,'uf':uf,'ano':ano,'mes':mes})
    # Um arquivo completo junto com partes pode duplicar o mesmo conteudo.
    for chave,partes in grupos.items():
        if None in partes and len(partes)>1: raise ValueError(f'Arquivo integral e partes coexistem para {chave}; revisar catalogo antes de baixar.')
        if None not in partes:
            numeros=sorted(int(p) for p in partes)
            if numeros != list(range(1,max(numeros)+1)): raise ValueError(f'Partes nao consecutivas: {chave}')
    faltantes=[f'RD{uf}{a%100:02}{m:02}' for a in ANOS for m in MESES for uf in UFS if (uf,a,m) not in grupos]
    return sorted(itens,key=lambda x:(x['ano'],x['mes'],x['uf'],x['nome'])),faltantes


def baixar(ftp,nome,destino):
    ftp.voidcmd('TYPE I')
    esperado=ftp.size(nome)
    parcial=destino.with_suffix('.dbc.part')
    n=0;ultimo=time.monotonic()
    with parcial.open('wb') as f:
        def receber(b):
            nonlocal n,ultimo
            f.write(b);n+=len(b)
            if time.monotonic()-ultimo>=5:
                print(f'  {nome}: {n/1024**2:.1f} MB',flush=True);ultimo=time.monotonic()
        ftp.retrbinary('RETR '+nome,receber,blocksize=1024**2)
    if n==0 or (esperado is not None and n!=esperado): raise IOError('DBC vazio/incompleto.')
    parcial.replace(destino)
    return n


def dbf_para_parquet(dbf,destino,item):
    tabela=DBF(str(dbf),raw=True,load=False,encoding='latin-1',ignore_missing_memofile=False)
    nomes=tabela.field_names
    extras=['_ARQUIVO_FONTE','_UF_ARQUIVO','_COMPETENCIA_ARQUIVO','_DBF_EXCLUIDO']
    if len(nomes)!=len(set(nomes)) or set(nomes)&set(extras): raise ValueError('Colunas duplicadas/conflitantes.')
    if not {'N_AIH','DT_INTER','DIAG_PRINC'}.issubset(nomes): raise ValueError('Layout nao corresponde a RD esperado.')
    if any(c.type not in ('C','N','F','D','L') for c in tabela.fields): raise ValueError('Tipo DBF nao suportado para preservacao textual.')
    esquema=pa.schema([(c,pa.string()) for c in nomes+extras]);parcial=destino.with_suffix('.parquet.part')
    total=excluidos=0;lote=[]
    with pq.ParquetWriter(parcial,esquema,compression='snappy') as w:
        for flag,registros in [('0',iter(tabela)),('1',iter(tabela.deleted))]:
            for r in registros:
                obj={c:r[c].decode('latin-1').strip() for c in nomes}
                obj.update(_ARQUIVO_FONTE=item['nome'],_UF_ARQUIVO=item['uf'],
                    _COMPETENCIA_ARQUIVO=f"{item['ano']}{item['mes']:02}",_DBF_EXCLUIDO=flag)
                lote.append(obj);total+=1;excluidos+=int(flag)
                if len(lote)>=LOTE: w.write_table(pa.Table.from_pylist(lote,schema=esquema));lote.clear()
        if lote:w.write_table(pa.Table.from_pylist(lote,schema=esquema))
    if total!=tabela.header.numrecords: raise ValueError(f'DBF declara {tabela.header.numrecords} registros; lidos {total}.')
    if pq.read_metadata(parcial).num_rows!=total: raise ValueError('Contagem Parquet divergente.')
    parcial.replace(destino)
    return {'registros':total,'registros_dbf_excluidos':excluidos,'colunas_originais':len(nomes),
        'layout_dbf':[{'nome':c.name,'tipo':c.type,'largura':c.length,'decimais':c.decimal_count} for c in tabela.fields]}


def main():
    for sub in ['brutos','parquet','temporarios']:(PASTA/sub).mkdir(parents=True,exist_ok=True)
    itens,faltantes=selecionar(catalogo())
    salvar_json(PASTA/'cobertura_catalogo.json',{'combinacoes_esperadas':len(ANOS)*len(MESES)*len(UFS),'arquivos_selecionados':len(itens),'combinacoes_nao_localizadas':faltantes,'recorte':'competencia de processamento'})
    if not itens: raise RuntimeError('Nenhum arquivo localizado no recorte.')
    manifesto=PASTA/'manifesto_download.json'
    estados=json.loads(manifesto.read_text(encoding='utf-8')) if manifesto.exists() else {}
    ftp=None;falhas=[];seguidas=0
    try:
        for i,item in enumerate(itens,1):
            nome=item['nome'];bruto=PASTA/'brutos'/nome;dbf=PASTA/'temporarios'/(Path(nome).stem+'.dbf');destino=PASTA/'parquet'/(Path(nome).stem+'.parquet')
            anterior=estados.get(nome,{})
            url=f'ftp://{HOST}{DIRETORIO}/{nome}'
            if anterior.get('status')=='concluido' and anterior.get('url')==url and destino.exists():
                try:
                    if sha256(destino)==anterior.get('sha256_parquet') and pq.read_metadata(destino).num_rows==anterior.get('registros'):
                        print(f'[{i}/{len(itens)}] {nome}: validado.',flush=True);seguidas=0;continue
                except Exception: pass
            print(f'[{i}/{len(itens)}] {nome}',flush=True)
            try:
                for tentativa in range(1,4):
                    try:
                        if ftp is None:ftp=conectar()
                        baixar(ftp,nome,bruto)
                        break
                    except Exception:
                        if ftp:ftp.close();ftp=None
                        if tentativa==3:raise
                        time.sleep(3*tentativa)
                # Remove somente o DBF temporario deste arquivo, nunca os dados finais.
                dbf.unlink(missing_ok=True)
                pyreaddbc.dbc2dbf(str(bruto),str(dbf))
                info=dbf_para_parquet(dbf,destino,item)
                estados[nome]={'status':'concluido','url':url,**item,**info,'coletado_utc':datetime.now(timezone.utc).isoformat(),'sha256_dbc':sha256(bruto),'sha256_parquet':sha256(destino)}
                dbf.unlink();seguidas=0
            except Exception as e:
                falhas.append(nome);seguidas+=1
                estados[nome]={'status':'erro','url':url,**item,'erro':str(e)}
                print(f'  Falha: {type(e).__name__}: {e}',flush=True)
            salvar_json(manifesto,estados)
            if seguidas>=3:
                print('Interrompido apos tres arquivos com falha consecutiva; reexecute para retomar.',flush=True);break
    finally:
        if ftp:ftp.close()
    pendentes=[x['nome'] for x in itens if estados.get(x['nome'],{}).get('status')!='concluido']
    salvar_json(PASTA/'status_coleta.json',{'arquivos_planejados':len(itens),'pendentes':pendentes,'nao_localizados':faltantes,'falhas_nesta_execucao':falhas})
    if pendentes or faltantes or falhas:raise RuntimeError(f'Coleta incompleta: {len(pendentes)} arquivos pendentes, {len(faltantes)} combinacoes nao localizadas. Veja status_coleta.json. Reexecute para retomar.')
    print('Concluido: '+str(PASTA/'parquet'))
    print('RD por competencia de processamento. Revisar AIHs e _DBF_EXCLUIDO antes de contar internacoes.')
    return estados


if __name__=='__main__':
    resultado_sih=main()


Consultando catalogo FTP (1/3)...
[1/1620] RDAC2001.dbc
[2/1620] RDAL2001.dbc
[3/1620] RDAM2001.dbc
[4/1620] RDAP2001.dbc
[5/1620] RDBA2001.dbc
[6/1620] RDCE2001.dbc
[7/1620] RDDF2001.dbc
[8/1620] RDES2001.dbc
[9/1620] RDGO2001.dbc
[10/1620] RDMA2001.dbc
[11/1620] RDMG2001.dbc
[12/1620] RDMS2001.dbc
[13/1620] RDMT2001.dbc
[14/1620] RDPA2001.dbc
[15/1620] RDPB2001.dbc
[16/1620] RDPE2001.dbc
[17/1620] RDPI2001.dbc
[18/1620] RDPR2001.dbc
[19/1620] RDRJ2001.dbc
[20/1620] RDRN2001.dbc
[21/1620] RDRO2001.dbc
[22/1620] RDRR2001.dbc
[23/1620] RDRS2001.dbc
[24/1620] RDSC2001.dbc
[25/1620] RDSE2001.dbc
[26/1620] RDSP2001.dbc
  RDSP2001.dbc: 10.5 MB
[27/1620] RDTO2001.dbc
[28/1620] RDAC2002.dbc
[29/1620] RDAL2002.dbc
[30/1620] RDAM2002.dbc
[31/1620] RDAP2002.dbc
[32/1620] RDBA2002.dbc
[33/1620] RDCE2002.dbc
[34/1620] RDDF2002.dbc
[35/1620] RDES2002.dbc
[36/1620] RDGO2002.dbc
[37/1620] RDMA2002.dbc
[38/1620] RDMG2002.dbc
[39/1620] RDMS2002.dbc
[40/1620] RDMT2002.dbc
[41/1620] RDPA2002.dbc
[42/1620